# RAG Implementations: Simple → Pro

This notebook walks through **two complete RAG pipelines** for the Qafela (قافلة) Expert Assistant:

| | Implementation | Pro Implementation |
|---|---|---|
| **Chunking** | Fixed-size (LangChain splitter) | LLM-powered smart chunks |
| **Retrieval** | Single vector search | Dual search (original + rewritten query) |
| **Re-ranking** | ❌ | ✅ LLM re-ranks results |
| **Query rewriting** | ❌ | ✅ LLM rewrites query from context |
| **Library** | LangChain | LangChain (`ChatOpenAI` + `ChatGroq`) + raw Chroma |

---

# 🟢 PART 1: Simple Implementation

A clean, LangChain-based RAG pipeline — straightforward and easy to understand.

### Flow:
```
Documents → Fixed-size chunks → HuggingFace Embeddings → Chroma VectorDB
Question  → Retrieve top-k chunks → Inject into System Prompt → LLM Answer
```

## Step 1.1 — Imports & Config

In [1]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

load_dotenv(override=True)

MODEL          = "llama3.2"                    # served by Ollama (ollama pull llama3.2)
OLLAMA_URL    = "http://localhost:11434/v1"  # Ollama's OpenAI-compatible endpoint
DB_NAME        = str(Path("..").resolve() / "vector_db")
KNOWLEDGE_BASE = str(Path("..").resolve() / "knowledge-base")
RETRIEVAL_K    = 10
REBUILD        = False  # True = rebuild both vector DBs (re-embeds everything); DBs that do not exist are always built

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # local, free, 384-dim


def simple_db_ready() -> bool:
    """True only if vector_db exists AND already holds vectors (an empty folder does not count)."""
    if not os.path.exists(DB_NAME):
        return False
    store = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
    return store._collection.count() > 0

print("✅ Config ready")

/var/folders/yr/p71lg5fd763fk09v9l9wj83w0000gn/T/ipykernel_79232/2571448524.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 14:31:41.217000 79232 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13129.11it/s]


✅ Config ready


## Step 1.2 — Load Documents from Knowledge Base

Each subfolder in `knowledge-base/` is a **document type** (e.g. `employees`, `products`, `contracts`).  
We load all `.md` files and tag each document with its `doc_type`.

In [2]:
def fetch_documents():
    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder,
            glob="**/*.md",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

documents = fetch_documents()
print(f"📄 Loaded {len(documents)} documents")
print(f"   Types: {set(d.metadata['doc_type'] for d in documents)}")

# Preview first document
print(f"\n--- First document preview ---")
print(f"Source: {documents[0].metadata['source']}")
print(documents[0].page_content[:300])

📄 Loaded 60 documents
   Types: {'employees', 'company', 'contracts', 'products'}

--- First document preview ---
Source: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/products/Rellm.md
# Product Summary
Rellm is an AI-powered Reinsurance Platform designed for B2B insurance carriers in the Egyptian market. Developed by Qafela, a leading InsurTech company, Rellm helps insurance companies offer better products at lower costs by providing a robust reinsurance solution. Leveraging mach


## Step 1.3 — Split into Chunks

`RecursiveCharacterTextSplitter` splits on natural boundaries (paragraphs → sentences → words).  
- `chunk_size=500` — max characters per chunk  
- `chunk_overlap=200` — overlap so context is not lost at boundaries

In [3]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    return text_splitter.split_documents(documents)

chunks = create_chunks(documents)
print(f"✂️  Split into {len(chunks)} chunks")
print(f"\n--- Example chunk ---")
print(chunks[0].page_content)

✂️  Split into 499 chunks

--- Example chunk ---
# Product Summary
Rellm is an AI-powered Reinsurance Platform designed for B2B insurance carriers in the Egyptian market. Developed by Qafela, a leading InsurTech company, Rellm helps insurance companies offer better products at lower costs by providing a robust reinsurance solution. Leveraging machine learning capabilities, Rellm enables insurers to analyze complex data, predict risks, and optimize their reinsurance strategies.


## Step 1.4 — Embed & Store in Chroma Vector DB

Each chunk is converted into a **384-dim embedding vector** using `all-MiniLM-L6-v2` (HuggingFace, runs locally)  
and stored in a local **Chroma** vector database for fast similarity search.

In [4]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )
    collection = vectorstore._collection
    count = collection.count()
    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"🗄️  VectorDB ready: {count:,} vectors with {dimensions:,} dimensions")
    return vectorstore

# ⚠️ This embeds every chunk — only runs if REBUILD is True or vector_db is missing/empty
if REBUILD or not simple_db_ready():
    vectorstore_simple = create_embeddings(chunks)
    print("✅ Ingestion complete")
else:
    print("⏭️  vector_db already has vectors — skipping ingestion (set REBUILD = True to rebuild)")

⏭️  vector_db already has vectors — skipping ingestion (set REBUILD = True to rebuild)


## Step 1.5 — Set Up Retriever & LLM

In [5]:
# Load existing vectorstore (skip re-ingestion if already done)
vectorstore_simple = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

retriever = vectorstore_simple.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL, base_url=OLLAMA_URL, api_key="ollama")

vector_count = vectorstore_simple._collection.count()
if vector_count == 0:
    raise RuntimeError("vector_db is empty — run the Step 1.4 cell (set REBUILD = True) before this cell")

print(f"🔍 Retriever ready ({vector_count:,} vectors)")
print("🤖 LLM ready")

🔍 Retriever ready (499 vectors)
🤖 LLM ready


## Step 1.6 — Test Retrieval

Verify the retriever finds relevant chunks for a test question.

In [6]:
test_question = "Who is Mohamed Elbahluan?"
retrieved_docs = retriever.invoke(test_question, k=RETRIEVAL_K)

print(f"🔍 Retrieved {len(retrieved_docs)} docs for: '{test_question}'\n")
for i, doc in enumerate(retrieved_docs[:3], 1):
    print(f"--- Doc {i} | source: {doc.metadata['source']} ---")
    print(doc.page_content[:200])
    print()

🔍 Retrieved 10 docs for: 'Who is Mohamed Elbahluan?'

--- Doc 1 | source: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md ---
```markdown
# Mohamed Elbahluan

## Summary
- Date of Birth: 1990-02-12
- Job Title: Co-Founder & CEO
- Location: Remote in Egypt
- Current Salary: EGP 5,000,000 base + EGP 1,500,000 bonus (2023)

## 

--- Doc 2 | source: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Karnak Life Partners for Healthllm.md ---
---

**Signatures:**

_________________________________
**Mohamed Elbahluan** | CEO | **Qafela (قافلة), Inc.** | Date: 15 March 2025

_________________________________
**Sherine Magdy** | CEO | **Karn

--- Doc 3 | source: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Rosetta Health Co. for Markellm.md ---
---

**Signatures:**

_________________________________
**Mohamed Elbahluan** | CEO | **Qafela (قافلة), Inc.** | Date: 17 M

## Step 1.7 — System Prompt & Answer Function

The retrieved chunks are injected into the system prompt as `{context}`.  
We also combine prior conversation history to make the assistant context-aware.

In [7]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Qafela (قافلة).
You are chatting with a user about Qafela (قافلة).
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def fetch_context(question: str) -> list[Document]:
    """Retrieve relevant context documents for a question."""
    return retriever.invoke(question, k=RETRIEVAL_K)

def combined_question(question: str, history: list[dict] = []) -> str:
    """Combine all the user's messages into a single string."""
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """Answer the given question with RAG; return the answer and the context documents."""
    combined = combined_question(question, history)
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs

print("✅ Answer function ready")

✅ Answer function ready


## Step 1.8 — Test the Simple RAG Pipeline

In [8]:
question = "Who is Mohamed Elbahluan?"
answer, context_docs = answer_question(question)

print(f"❓ Question: {question}\n")
print(f"💬 Answer:\n{answer}\n")
print(f"📚 Context sources used:")
for doc in context_docs:
    print(f"   - {doc.metadata['source']}")

❓ Question: Who is Mohamed Elbahluan?

💬 Answer:
Mohamed Elbahluan is the Co-Founder and CEO of Qafela (قافلة), a company that aims to empower insurance innovation in Egypt. He has a strong background in the financial industry, having worked as an Investment Banking Analyst at EFG Hermes and a Risk Management Analyst at Arab Bank, before moving on to roles in Business Development and then founding Qafela in 2015.

📚 Context sources used:
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Karnak Life Partners for Healthllm.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Rosetta Health Co. for Markellm.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with NileShield Insurance for Homellm.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-b

In [9]:
# Try another question with history (multi-turn)
history = [
    {"role": "user", "content": "Who is Mohamed Elbahluan?"},
    {"role": "assistant", "content": answer}
]

followup = "What products did he launch?"
answer2, _ = answer_question(followup, history)

print(f"❓ Follow-up: {followup}\n")
print(f"💬 Answer:\n{answer2}")

❓ Follow-up: What products did he launch?

💬 Answer:
I don't have information on the specific products launched by Mohamed Elbahluan as the CEO of Qafela (قافلة). The provided context only mentions his career progression and the founding of the company, but does not provide details on the products or services offered by Qafela. If you're looking for information on Qafela's products or services, I'd be happy to try and help you find that information.


---

# 🔴 PART 2: Pro Implementation

A production-grade RAG pipeline with **LLM-powered chunking**, **query rewriting**, **dual retrieval**, and **LLM re-ranking**.

### Flow:
```
INGEST:
  Documents → LLM splits into smart chunks (headline + summary + original_text)
           → HuggingFace Embeddings → Raw ChromaDB (preprocessed_db)

ANSWER:
  Question → LLM rewrites query (context-aware)
          → Dual vector search (original + rewritten)
          → Merge results
          → LLM re-ranks by relevance
          → Top-K chunks → System Prompt → LLM Answer
```

## Step 2.1 — Imports & Config

In [10]:
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from langchain_openai import ChatOpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from tenacity import retry, wait_exponential, stop_after_attempt
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)

INGEST_MODEL       = "gpt-4.1-nano"          # LLM chunking (pro ingest), OpenAI API
# PRO_MODEL = "openai/gpt-oss-120b"   # Groq option (free tier: 200k tokens/day)
PRO_MODEL          = "gpt-4.1-nano"        # rewrite, re-rank and answer (OpenAI API)
PRO_DB_NAME        = str(Path("..").resolve() / "preprocessed_db")
KNOWLEDGE_BASE_PATH = Path("..").resolve() / "knowledge-base"
COLLECTION_NAME    = "docs"
AVERAGE_CHUNK_SIZE = 100   # used to estimate how many chunks to create per doc
WORKERS            = 3     # parallel workers for LLM chunking (set to 1 if rate limited)
PRO_RETRIEVAL_K    = 20    # how many chunks to fetch from DB
FINAL_K            = 10    # how many to keep after re-ranking
wait               = wait_exponential(multiplier=1, min=10, max=240)


def pro_db_ready() -> bool:
    """True only if preprocessed_db has a non-empty collection (an empty folder does not count)."""
    if not os.path.exists(PRO_DB_NAME):
        return False
    try:
        return PersistentClient(path=PRO_DB_NAME).get_collection(COLLECTION_NAME).count() > 0
    except Exception:
        return False


ingest_llm = ChatOpenAI(model=INGEST_MODEL)   # used to chunk documents (reads OPENAI_API_KEY from .env)
pro_llm    = ChatOpenAI(model=PRO_MODEL)      # used to rewrite, re-rank and answer


print(f"✅ Pro config ready (pro steps: {type(pro_llm).__name__} / {PRO_MODEL}, chunking: {INGEST_MODEL})")

✅ Pro config ready (pro steps: ChatOpenAI / gpt-4.1-nano, chunking: gpt-4.1-nano)


## Step 2.2 — Data Models (Pydantic)

We use Pydantic models to enforce **structured output** from the LLM.

Each `Chunk` produced by the LLM has:
- **`headline`** — a short label (what queries might match this)
- **`summary`** — a few sentences about the chunk's content  
- **`original_text`** — the raw text from the document, unchanged

This is stored as `headline + summary + original_text` — richer than raw text alone.

In [11]:
class Result(BaseModel):
    """A retrieved chunk with its metadata."""
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    """A single LLM-generated chunk with headline, summary, and original text."""
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        """Convert to a Result combining headline + summary + original_text."""
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
        )

class Chunks(BaseModel):
    """A list of chunks returned by the LLM."""
    chunks: list[Chunk]

class RankOrder(BaseModel):
    """Re-ranked order of chunk IDs from most to least relevant."""
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

print("✅ Pydantic models defined")

✅ Pydantic models defined


## Step 2.3 — Load Documents (Homemade Loader)

Instead of LangChain's `DirectoryLoader`, we use a simple homemade loader.

In [12]:
def pro_fetch_documents():
    """A homemade version of the LangChain DirectoryLoader."""
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({
                    "type": doc_type,
                    "source": file.as_posix(),
                    "text": f.read()
                })
    print(f"📄 Loaded {len(documents)} documents")
    return documents

pro_documents = pro_fetch_documents()

# Preview
print(f"\n--- First document ---")
print(f"Type:   {pro_documents[0]['type']}")
print(f"Source: {pro_documents[0]['source']}")
print(pro_documents[0]['text'][:300])

📄 Loaded 60 documents

--- First document ---
Type:   products
Source: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/products/Rellm.md
# Product Summary
Rellm is an AI-powered Reinsurance Platform designed for B2B insurance carriers in the Egyptian market. Developed by Qafela, a leading InsurTech company, Rellm helps insurance companies offer better products at lower costs by providing a robust reinsurance solution. Leveraging mach


## Step 2.4 — LLM-Powered Smart Chunking

Instead of blindly splitting by character count, we ask the **LLM itself** to read each document and produce semantically meaningful chunks with headlines and summaries.

The prompt estimates how many chunks are needed based on document length.

In [13]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Qafela (قافلة).
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

def make_messages(document):
    return [{"role": "user", "content": make_prompt(document)}]

print("✅ Prompt builder ready")

# Preview the prompt for one document
print("\n--- Prompt preview (first 500 chars) ---")
print(make_prompt(pro_documents[30])[:500])

✅ Prompt builder ready

--- Prompt preview (first 500 chars) ---

You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Qafela (قافلة).
The document is of type: employees
The document has been retrieved from: /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Salma Ibrahim.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is return


In [14]:
@retry(wait=wait, stop=stop_after_attempt(5))
def process_document(document):
    """Ask the LLM to chunk one document into structured Chunk objects."""
    messages = make_messages(document)
    doc_as_chunks = ingest_llm.with_structured_output(Chunks).invoke(messages).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

# Test on a single document first
print("🧪 Testing LLM chunking on one document...")
test_chunks = process_document(pro_documents[30])
print(f"✅ Got {len(test_chunks)} chunks from the LLM\n")
print("--- First chunk ---")
print(test_chunks[0].page_content)


🧪 Testing LLM chunking on one document...
✅ Got 6 chunks from the LLM

--- First chunk ---
Introduction to Salma Ibrahim

This section introduces Salma Ibrahim, highlighting her personal details, current job role, and location.

# Salma Ibrahim

## Summary
- Date of Birth: 1992-02-15
- Job Title: Head of Product
- Location: Remote in Egypt
- Current Salary: EGP 250,000 base + EGP 75,000 bonus


In [15]:
print(test_chunks[0].page_content)


Introduction to Salma Ibrahim

This section introduces Salma Ibrahim, highlighting her personal details, current job role, and location.

# Salma Ibrahim

## Summary
- Date of Birth: 1992-02-15
- Job Title: Head of Product
- Location: Remote in Egypt
- Current Salary: EGP 250,000 base + EGP 75,000 bonus


In [16]:
print(test_chunks[1].page_content)

Salma Ibrahim's Career at Qafela (قافلة) and Previous Roles

Describes Salma Ibrahim's career progression within Qafela and her previous positions at other companies.

## Qafela (قافلة) Career Progression
- 2018 - Present: Head of Product at Qafela (قافلة) — key achievements
  - 2017: Senior Product Manager at Al Ahram Bank
  - 2016: Product Analyst at Alexandria Insurance Co.
  - 2015: Junior Product Analyst at Heliopolis Insurance Co.


In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def pro_create_chunks(documents):
    """
    Create smart chunks with a thread pool.
    A process pool cannot call a function defined in a notebook on macOS.
    If you get rate limit errors, set WORKERS = 1 above.
    """
    chunks = []
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(process_document, document) for document in documents]
        for future in tqdm(as_completed(futures), total=len(futures)):
            chunks.extend(future.result())
    return chunks

# ⚠️ This makes many API calls (one per document) — only runs if REBUILD is True or preprocessed_db is missing/empty
if REBUILD or not pro_db_ready():
    print("🔄 Creating smart chunks for all documents (this may take a few minutes)...")
    pro_chunks = pro_create_chunks(pro_documents)
    print(f"✅ Total chunks created: {len(pro_chunks)}")
else:
    print("⏭️  preprocessed_db already has vectors — skipping chunking (set REBUILD = True to rebuild)")

⏭️  preprocessed_db already has vectors — skipping chunking (set REBUILD = True to rebuild)


## Step 2.5 — Embed & Store in Raw Chroma DB

We use the raw `chromadb` client (no LangChain wrapper) for more control.

In [20]:
def pro_create_embeddings(chunks, batch_size=100):
    chroma = PersistentClient(path=PRO_DB_NAME)

    # Clear existing collection if present
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)

    texts   = [chunk.page_content for chunk in chunks]
    metas   = [chunk.metadata for chunk in chunks]

    # Batch embed all chunks
    print("🔢 Generating embeddings...")
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        vectors.extend(embeddings.embed_documents(batch))

    collection = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"🗄️  Pro VectorDB ready: {collection.count()} documents stored")

# ⚠️ Only runs if REBUILD is True or preprocessed_db is missing/empty
if REBUILD or not pro_db_ready():
    pro_create_embeddings(pro_chunks)
    print("✅ Pro ingestion complete")
else:
    print("⏭️  preprocessed_db already has vectors — skipping ingestion (set REBUILD = True to rebuild)")

⏭️  preprocessed_db already has vectors — skipping ingestion (set REBUILD = True to rebuild)


## Step 2.6 — Connect to Pro VectorDB

In [21]:
pro_chroma     = PersistentClient(path=PRO_DB_NAME)
pro_collection = pro_chroma.get_or_create_collection(COLLECTION_NAME)

if pro_collection.count() == 0:
    raise RuntimeError("preprocessed_db is empty — run the Step 2.4 and 2.5 cells first")
print(f"🗄️  Connected to Pro VectorDB: {pro_collection.count()} chunks available")

🗄️  Connected to Pro VectorDB: 340 chunks available


## Step 2.7 — Query Rewriting

Before searching, the LLM **rewrites the user's question** into a more specific, context-aware query.  
This handles follow-up questions like "What role does she have?" (who is "she"?)

In [22]:
@retry(wait=wait, stop=stop_after_attempt(5))
def rewrite_query(question, history=None):
    """Rewrite the user's question to be more specific and context-aware."""
    if history is None:
        history = []

    message = f"""
You are in a conversation with a user.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Since the conversation is contextual, understand the meaning of the user question and add details based on the history.
Condense everything in a single contextually-rich VERY short and specific question, most likely to surface content.

EXAMPLE:
user: Who is the founder? -> Query: who is the founder?
assistant: The founder is FooBar
user: What role covers? -> Query: What role FooBar covers?
...

IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
"""
    response = pro_llm.invoke([{"role": "system", "content": message}])
    return response.content

# Test query rewriting
original = "What about his products?"
history_example = [
    {"role": "user", "content": "Who is Mohamed Elbahluan?"},
    {"role": "assistant", "content": "Mohamed Elbahluan is the Co-Founder and CEO of Qafela (قافلة)."}
]
rewritten = rewrite_query(original, history_example)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")

Original : What about his products?
Rewritten: What products does Mohamed Elbahluan's company Qafela offer?


## Step 2.8 — Dual Retrieval + Merge

We run **two** vector searches — one with the original question, one with the rewritten query — then merge unique results.

In [23]:
def fetch_context_unranked(question):
    """Run a single vector search and return Result objects."""
    query_vec = embeddings.embed_query(question)

    results = pro_collection.query(query_embeddings=[query_vec], n_results=PRO_RETRIEVAL_K)
    chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks

def merge_chunks(chunks, reranked):
    """Merge two result lists, keeping unique chunks only."""
    merged = chunks[:]
    existing = [chunk.page_content for chunk in chunks]
    for chunk in reranked:
        if chunk.page_content not in existing:
            merged.append(chunk)
    return merged

# Test dual retrieval
q = "Who is Mohamed Elbahluan?"
chunks1 = fetch_context_unranked(q)
chunks2 = fetch_context_unranked(rewrite_query(q))
merged  = merge_chunks(chunks1, chunks2)

print(f"Original query results : {len(chunks1)}")
print(f"Rewritten query results: {len(chunks2)}")
print(f"Merged (unique)        : {len(merged)}")

Original query results : 20
Rewritten query results: 20
Merged (unique)        : 20


## Step 2.9 — LLM Re-ranking

After retrieval, we ask the LLM to **re-order** the chunks by relevance to the question.  
This goes beyond pure vector similarity — the LLM understands the question semantically.

In [24]:
@retry(wait=wait, stop=stop_after_attempt(5))
def rerank(question, chunks):
    """Ask the LLM to re-rank chunks by relevance to the question."""
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    order = pro_llm.with_structured_output(RankOrder).invoke(messages).order
    # keep valid, unique ids only; append any chunks the LLM left out
    ranked = []
    for i in order:
        if 1 <= i <= len(chunks) and i not in ranked:
            ranked.append(i)
    ranked += [i for i in range(1, len(chunks) + 1) if i not in ranked]
    return [chunks[i - 1] for i in ranked]

# Test re-ranking
reranked = rerank(q, merged)
final    = reranked[:FINAL_K]
print(f"After re-ranking, top {FINAL_K} sources:")
for c in final:
    print(f"  - {c.metadata['source']}")

After re-ranking, top 10 sources:
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/company/about.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Karnak Life Partners for Healthllm.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/contracts/Contract with Pharaoh Auto Insurance for Bizllm.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Hassan Mahmoud.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mahmoud Zaki.md
  - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Ahmed Fathy.md
  - /Users/bahlu/Desktop/Bahlu/le

## Step 2.10 — Full `pro_fetch_context` Pipeline

Combines query rewriting → dual retrieval → merge → re-rank → top-K.

In [25]:
def pro_fetch_context(original_question, history=None):
    """Full pro retrieval pipeline."""
    rewritten_question = rewrite_query(original_question, history)
    print(f"  📝 Rewritten query: {rewritten_question}")

    chunks1 = fetch_context_unranked(original_question)
    chunks2 = fetch_context_unranked(rewritten_question)
    chunks  = merge_chunks(chunks1, chunks2)
    print(f"  🔍 Retrieved {len(chunks)} unique chunks (dual search)")

    reranked = rerank(original_question, chunks)
    top_k    = reranked[:FINAL_K]
    print(f"  🏆 Re-ranked, keeping top {FINAL_K}")
    return top_k

print("✅ pro_fetch_context pipeline ready")

✅ pro_fetch_context pipeline ready


## Step 2.11 — Pro Answer Function

In [26]:
PRO_SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Qafela (قافلة).
You are chatting with a user about Qafela (قافلة).
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks
    )
    system_prompt = PRO_SYSTEM_PROMPT.format(context=context)
    return (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )

@retry(wait=wait, stop=stop_after_attempt(5))
def answer_question_pro(question: str, history: list[dict] = []) -> tuple[str, list]:
    """Answer a question using the full Pro RAG pipeline."""
    chunks   = pro_fetch_context(question, history)
    messages = make_rag_messages(question, history, chunks)
    response = pro_llm.invoke(messages)
    return response.content, chunks

print("✅ Pro answer function ready")

✅ Pro answer function ready


## Step 2.12 — Test the Pro RAG Pipeline

In [27]:
question = "Who is Mohamed Elbahluan?"

print(f"❓ Question: {question}\n")
answer_pro, context_chunks = answer_question_pro(question)

print(f"\n💬 Answer:\n{answer_pro}\n")
print(f"📚 Context sources used:")
for chunk in context_chunks:
    print(f"   - {chunk.metadata['source']}")

❓ Question: Who is Mohamed Elbahluan?

  📝 Rewritten query: Who is Mohamed Elbahluan?
  🔍 Retrieved 20 unique chunks (dual search)
  🏆 Re-ranked, keeping top 10

💬 Answer:
Mohamed Elbahluan is the Co-Founder and CEO of Qafela (قافلة). He was born on February 12, 1990, and is based remotely in Egypt. Mohamed founded Qafela in 2015 in Cairo with the aim to lead in InsurTech within Egypt. He has a background in business development, investment banking, and risk management, having worked at Citadel Capital, EFG Hermes, and Arab Bank before founding Qafela. His current salary for 2023 is EGP 5,000,000 base plus EGP 1,500,000 bonus.

📚 Context sources used:
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
   - /Users/bahlu/Desktop/Bahlu/learning/insurellm-rag/knowledge-base/employees/Mohamed Elbahluan.md
   - /Users/bahlu/Desktop/Bahlu/learn

In [28]:
# Multi-turn test — follow-up question
history = [
    {"role": "user",      "content": "Who is Mohamed Elbahluan?"},
    {"role": "assistant", "content": answer_pro}
]

followup = "What products did he launch?"
print(f"❓ Follow-up: {followup}\n")
answer_pro2, _ = answer_question_pro(followup, history)
print(f"💬 Answer:\n{answer_pro2}")

❓ Follow-up: What products did he launch?

  📝 Rewritten query: What insurance products has Mohamed Elbahluan launched through Qafela?
  🔍 Retrieved 36 unique chunks (dual search)
  🏆 Re-ranked, keeping top 10
💬 Answer:
Mohamed Elbahluan, as the founder and CEO of Qafela, has been instrumental in establishing the company's product portfolio, which focuses on innovative insurance solutions. The key products launched by Qafela include Carllm, Homellm, and Healthllm. Carllm was launched in 2018 and quickly gained traction in the Egyptian insurance market. Subsequently, in 2020, Qafela expanded its product offerings to include Homellm and Healthllm, further strengthening its position as a leading InsurTech company in Egypt.


---

# 📊 Summary: Simple vs Pro

| Step | Simple Implementation | Pro Implementation |
|------|----------------------|--------------------|
| **Document loading** | LangChain `DirectoryLoader` | Homemade loader |
| **Chunking** | Fixed-size (`RecursiveCharacterTextSplitter`) | LLM-generated (headline + summary + text) |
| **Chunk quality** | Raw text only | Rich: headline + summary + original |
| **Parallelism** | Single-threaded | Thread pool (3 workers) |
| **Vector DB** | LangChain `Chroma` wrapper | Raw `chromadb.PersistentClient` |
| **Query** | Raw user question | LLM rewrites question first |
| **Retrieval** | One vector search | Two searches (original + rewritten) |
| **Result merging** | N/A | Merges unique chunks from both searches |
| **Re-ranking** | ❌ | ✅ LLM re-ranks by relevance |
| **Retry logic** | ❌ | ✅ Exponential backoff on all LLM calls |
| **Model** | `gpt-4.1-nano` via `ChatOpenAI` | `gpt-4.1-nano` (chunking) + `openai/gpt-oss-120b` via `ChatGroq` (rewrite, re-rank, answer) |
| **Best for** | Learning, prototyping | Production use |


---

# Compare Simple vs Pro

Same logic as the course `evaluation/eval.py`, run on `evaluation/tests.jsonl` for both pipelines.

For each question:

- **Retrieval:** MRR, nDCG, and keyword coverage of the chunks returned by `fetch_context` / `pro_fetch_context`
- **Answer:** an LLM judge (`gpt-4.1-nano`) scores accuracy, completeness, and relevance from 1 to 5 against `reference_answer`

Set `MAX_TESTS` below. Each test calls retrieval and the judge for both pipelines, so start with a few tests.


In [30]:
import json
import math
from pathlib import Path
from pydantic import BaseModel, Field

TESTS_PATH = Path("..").resolve() / "evaluation" / "tests.jsonl"
EVAL_MODEL = "gpt-4.1-nano"
MAX_TESTS = 100   # raise this after a short run looks right

judge_llm = ChatOpenAI(model=EVAL_MODEL)


class TestQuestion(BaseModel):
    """A test question with expected keywords and reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., direct_fact, spanning, temporal)")


def load_tests() -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    tests = []
    with open(TESTS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests


class RetrievalEval(BaseModel):
    """Evaluation metrics for retrieval performance."""

    mrr: float = Field(description="Mean Reciprocal Rank - average across all keywords")
    ndcg: float = Field(description="Normalized Discounted Cumulative Gain (binary relevance)")
    keywords_found: int = Field(description="Number of keywords found in top-k results")
    total_keywords: int = Field(description="Total number of keywords to find")
    keyword_coverage: float = Field(description="Percentage of keywords found")


class AnswerEval(BaseModel):
    """LLM-as-a-judge evaluation of answer quality."""

    feedback: str = Field(
        description="Concise feedback on the answer quality, comparing it to the reference answer and evaluating based on the retrieved context"
    )
    accuracy: float = Field(
        description="How factually correct is the answer compared to the reference answer? 1 (wrong. any wrong answer must score 1) to 5 (ideal - perfectly accurate). An acceptable answer would score 3."
    )
    completeness: float = Field(
        description="How complete is the answer in addressing all aspects of the question? 1 (very poor - missing key information) to 5 (ideal - all the information from the reference answer is provided completely). Only answer 5 if ALL information from the reference answer is included."
    )
    relevance: float = Field(
        description="How relevant is the answer to the specific question asked? 1 (very poor - off-topic) to 5 (ideal - directly addresses question and gives no additional information). Only answer 5 if the answer is completely relevant to the question and gives no additional information."
    )


def calculate_mrr(keyword: str, retrieved_docs: list) -> float:
    """Calculate reciprocal rank for a single keyword (case-insensitive)."""
    keyword_lower = keyword.lower()
    for rank, doc in enumerate(retrieved_docs, start=1):
        if keyword_lower in doc.page_content.lower():
            return 1.0 / rank
    return 0.0


def calculate_dcg(relevances: list[int], k: int) -> float:
    """Calculate Discounted Cumulative Gain."""
    dcg = 0.0
    for i in range(min(k, len(relevances))):
        dcg += relevances[i] / math.log2(i + 2)  # i+2 because rank starts at 1
    return dcg


def calculate_ndcg(keyword: str, retrieved_docs: list, k: int = 10) -> float:
    """Calculate nDCG for a single keyword (binary relevance, case-insensitive)."""
    keyword_lower = keyword.lower()

    # Binary relevance: 1 if keyword found, 0 otherwise
    relevances = [
        1 if keyword_lower in doc.page_content.lower() else 0 for doc in retrieved_docs[:k]
    ]

    # DCG
    dcg = calculate_dcg(relevances, k)

    # Ideal DCG (best case: keyword in first position)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = calculate_dcg(ideal_relevances, k)

    return dcg / idcg if idcg > 0 else 0.0


def evaluate_retrieval(test: TestQuestion, fetch_context_fn, k: int = 10) -> RetrievalEval:
    """Evaluate retrieval performance for a test question (fetch_context_fn picks the pipeline)."""
    retrieved_docs = fetch_context_fn(test.question)

    # Calculate MRR (average across all keywords)
    mrr_scores = [calculate_mrr(keyword, retrieved_docs) for keyword in test.keywords]
    avg_mrr = sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0

    # Calculate nDCG (average across all keywords)
    ndcg_scores = [calculate_ndcg(keyword, retrieved_docs, k) for keyword in test.keywords]
    avg_ndcg = sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0

    # Calculate keyword coverage
    keywords_found = sum(1 for score in mrr_scores if score > 0)
    total_keywords = len(test.keywords)
    keyword_coverage = (keywords_found / total_keywords * 100) if total_keywords > 0 else 0.0

    return RetrievalEval(
        mrr=avg_mrr,
        ndcg=avg_ndcg,
        keywords_found=keywords_found,
        total_keywords=total_keywords,
        keyword_coverage=keyword_coverage,
    )


def evaluate_answer(test: TestQuestion, answer_question_fn) -> tuple[AnswerEval, str, list]:
    """Evaluate answer quality using LLM-as-a-judge (answer_question_fn picks the pipeline)."""
    generated_answer, retrieved_docs = answer_question_fn(test.question)

    # LLM judge prompt
    judge_messages = [
        {
            "role": "system",
            "content": "You are an expert evaluator assessing the quality of answers. Evaluate the generated answer by comparing it to the reference answer. Only give 5/5 scores for perfect answers.",
        },
        {
            "role": "user",
            "content": f"""Question:
{test.question}

Generated Answer:
{generated_answer}

Reference Answer:
{test.reference_answer}

Please evaluate the generated answer on three dimensions:
1. Accuracy: How factually correct is it compared to the reference answer? Only give 5/5 scores for perfect answers.
2. Completeness: How thoroughly does it address all aspects of the question, covering all the information from the reference answer?
3. Relevance: How well does it directly answer the specific question asked, giving no additional information?

Provide detailed feedback and scores from 1 (very poor) to 5 (ideal) for each dimension. If the answer is wrong, then the accuracy score must be 1.""",
        },
    ]

    # Call LLM judge with structured output
    answer_eval = judge_llm.with_structured_output(AnswerEval).invoke(judge_messages)

    return answer_eval, generated_answer, retrieved_docs


@retry(wait=wait, stop=stop_after_attempt(5))
def answer_from_docs(question, docs):
    """Answer with already-retrieved pro chunks (same prompt as answer_question_pro, no second retrieval)."""
    return pro_llm.invoke(make_rag_messages(question, [], docs)).content, docs


tests = load_tests()[:MAX_TESTS]
print(f"Scoring {len(tests)} tests from {TESTS_PATH.name}\n")

rows = []
for index, test in enumerate(tests, start=1):
    print(f"[{index}/{len(tests)}] {test.category}: {test.question}")

    simple_retrieval = evaluate_retrieval(test, fetch_context)
    pro_docs         = pro_fetch_context(test.question)   # one rewrite + re-rank per test, reused below (saves tokens)
    pro_retrieval    = evaluate_retrieval(test, lambda _q: pro_docs)
    simple_judge, simple_answer, _ = evaluate_answer(test, answer_question)
    pro_judge, pro_answer, _       = evaluate_answer(test, lambda q: answer_from_docs(q, pro_docs))

    rows.append({
        "simple_mrr": simple_retrieval.mrr,
        "pro_mrr": pro_retrieval.mrr,
        "simple_ndcg": simple_retrieval.ndcg,
        "pro_ndcg": pro_retrieval.ndcg,
        "simple_coverage": simple_retrieval.keyword_coverage,
        "pro_coverage": pro_retrieval.keyword_coverage,
        "simple_accuracy": simple_judge.accuracy,
        "pro_accuracy": pro_judge.accuracy,
        "simple_completeness": simple_judge.completeness,
        "pro_completeness": pro_judge.completeness,
        "simple_relevance": simple_judge.relevance,
        "pro_relevance": pro_judge.relevance,
    })
    print(
        f"  retrieval  simple MRR {simple_retrieval.mrr:.2f} coverage {simple_retrieval.keyword_coverage:.0f}%"
        f"   |  pro MRR {pro_retrieval.mrr:.2f} coverage {pro_retrieval.keyword_coverage:.0f}%"
    )
    print(
        f"  answer     simple {simple_judge.accuracy:.1f}/{simple_judge.completeness:.1f}/{simple_judge.relevance:.1f}"
        f"   |  pro {pro_judge.accuracy:.1f}/{pro_judge.completeness:.1f}/{pro_judge.relevance:.1f}"
        "   (accuracy/completeness/relevance)"
    )


def avg(key):
    return sum(row[key] for row in rows) / len(rows)


print("\n" + "=" * 58)
print(f"{'metric':<24} {'simple':>12} {'pro':>12}")
print("-" * 58)
for label, simple_key, pro_key, as_percent in [
    ("retrieval MRR", "simple_mrr", "pro_mrr", False),
    ("retrieval nDCG", "simple_ndcg", "pro_ndcg", False),
    ("keyword coverage", "simple_coverage", "pro_coverage", True),
    ("answer accuracy", "simple_accuracy", "pro_accuracy", False),
    ("answer completeness", "simple_completeness", "pro_completeness", False),
    ("answer relevance", "simple_relevance", "pro_relevance", False),
]:
    left, right = avg(simple_key), avg(pro_key)
    if as_percent:
        print(f"{label:<24} {left:>11.1f}% {right:>11.1f}%")
    else:
        print(f"{label:<24} {left:>12.2f} {right:>12.2f}")
print("=" * 58)
print(f"Tests compared: {len(rows)}")


Scoring 100 tests from tests.jsonl

[1/100] direct_fact: What is the job title of Ahmed Fathy?
  📝 Rewritten query: What is the job title of Ahmed Fathy?
  🔍 Retrieved 20 unique chunks (dual search)
  🏆 Re-ranked, keeping top 10
  retrieval  simple MRR 1.00 coverage 100%   |  pro MRR 1.00 coverage 100%
  answer     simple 5.0/4.0/5.0   |  pro 5.0/5.0/5.0   (accuracy/completeness/relevance)
[2/100] direct_fact: What is the location where Ahmed Fathy worked as Financial Manager at Arab Bank Egypt?
  📝 Rewritten query: Where did Ahmed Fathy work as Financial Manager at Arab Bank Egypt?
  🔍 Retrieved 22 unique chunks (dual search)
  🏆 Re-ranked, keeping top 10
  retrieval  simple MRR 1.00 coverage 100%   |  pro MRR 0.50 coverage 100%
  answer     simple 5.0/5.0/5.0   |  pro 5.0/4.0/5.0   (accuracy/completeness/relevance)
[3/100] direct_fact: What is the product name of Qafela (قافلة) that Ahmed Fathy led the transition of to a new ERP system?
  📝 Rewritten query: What is the product name o